[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/05_gap_characterization.ipynb)

# R2-Q1 Week 4 — Characterize the PV → PD transfer gap

This is the last classroom notebook for R2-Q1. It picks up where
Notebook 04 left off.

Notebook 04 produced a headline number: your PV-trained classifier
loses some amount of accuracy when evaluated on PlantDoc rather than
PlantVillage, and the bootstrap confidence interval told you whether
that gap was real. What N04 did not tell you is *where* the gap lives
— which host species the model transfers cleanly to, which disease
categories it handles poorly, and which combinations drive most of
the failure.

This notebook decomposes the gap into two cuts:

1. **By host group.** Per-host accuracy across all disease categories.
   Answers: which plants does the model recognize cleanly when it
   sees them in a field photograph, and which does it stumble on?
2. **By disease category.** Per-category accuracy across all hosts.
   Answers: which disease categories transfer from lab to field, and
   which ones the model evidently learned in a PV-specific way?

By the end of this notebook you will have:

- A flat decomposition table (`gap_decomposition.parquet`) with one
  row per group: host or category name, sample count, accuracy, 95%
  bootstrap CI, and a flag for groups with too few samples to support
  a reliable confidence interval.
- A reader-facing summary (`characterization_summary.json`) naming
  the hosts and categories driving the gap and listing which cuts
  had enough data to be reliable.
- Two bar charts (`gap_by_host.png`, `gap_by_category.png`) with
  bootstrap error bars — the figures your Week 5 paper will lean on.

### What you need from N04

This notebook reads `pd_predictions.parquet` from your R2-Q1 output
directory (along with `precommit.json` for the disease-category
mapping). Both should already be there from running N04. Section 2
checks for them and fails with a clear error if either is missing.

### Install the iRI Lab package

If you opened this notebook in a fresh Colab session, you need to
install the package first. If you just finished running Notebook 04
in the same session, the package is already installed and you only
need to refresh it.

The diagnostic cell below tells you which of the two install patterns
in the following cell to uncomment. Run the diagnostic, read what it
prints, then uncomment exactly one line in the cell after it.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026 (no GPU required for this notebook),
# seed everything, and point OUTPUT_DIR at the R2-Q1 output directory.
import irilab2026 as iri

iri.setup(gpu_required=False)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

### Development mode

This notebook is fast — the per-group bootstrap is dataframe-only and
runs in seconds even at the precommit's full iteration count.

The `DEV_MODE` switch below still earns its keep on small iterations.
If you're tweaking a plot, fixing a typo, or debugging a save path,
clipping the bootstrap from a thousand iterations down to a hundred
shaves off most of the wait. The pattern matches the toggle you used
in Notebooks 03 and 04 — same shape, one knob fewer.

- **`DEV_MODE = True`**: cap the bootstrap at 100 iterations regardless
  of your precommit. The point estimates are still real, but the
  confidence intervals are wider than they would be at the precommit's
  full count. Numbers produced in dev mode are not paper-ready.
- **`DEV_MODE = False`**: honor your precommit's `n_bootstrap`. These
  are the numbers you report.

Ship the notebook with `DEV_MODE = False` so a student opening it
fresh produces real numbers. Flip to `True` only while you're
iterating on something.

In [ ]:
### 1.1) DEV_MODE switch

DEV_MODE = False

if DEV_MODE:
    BOOTSTRAP_CAP = 100      # max bootstrap iterations regardless of precommit
    print("⚠️  DEV_MODE is ON — numbers produced are NOT paper-ready.")
    print(f"    Bootstrap cap: {BOOTSTRAP_CAP} iterations")
else:
    BOOTSTRAP_CAP = None     # honor precommit's n_bootstrap
    print("DEV_MODE is OFF — full production run.")
    print(f"    Bootstrap cap: honor precommit n_bootstrap")

## 2) Read your inputs and pre-commits

This notebook reads two files produced by earlier work in R2-Q1:

- **`pd_predictions.parquet`** — N04's per-image predictions on
  PlantDoc's test split, with `host` and `true_category` metadata
  already joined in. Every row of analysis in this notebook starts
  from this table.
- **`precommit.json`** — your Week 1 methodology commitments. This
  notebook honors the bootstrap iteration count and significance
  level from there. It also reads `categories_used` so your bar
  charts order categories the same way the rest of R2-Q1 has.

If either file is missing from your R2-Q1 output directory, the
cells below fail with a message pointing you back to the notebook
that produces it. You can't run this notebook without both.

In [ ]:
### 2.1) Read precommit.json
import json

precommit_path = OUTPUT_DIR / "precommit.json"

if not precommit_path.exists():
    raise FileNotFoundError(
        f"Couldn't find {precommit_path}. "
        "Run Notebook 02 (`02_pd_orientation.ipynb`) to completion — "
        "the last section writes precommit.json to this location."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# Validate the three top-level keys exist. Each section below
# cherry-picks the specific fields it needs from these.
expected_keys = {"aggregation_level", "class_space_remapping", "statistical_test"}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing these top-level keys: {sorted(missing)}. "
        f"Re-run Notebook 02 to regenerate the file."
    )

print(f"Loaded: {precommit_path}")
print(f"  Top-level keys: {sorted(precommit.keys())}")

In [ ]:
### 2.2) Read pd_predictions.parquet
import pandas as pd

pd_predictions_path = OUTPUT_DIR / "pd_predictions.parquet"

if not pd_predictions_path.exists():
    raise FileNotFoundError(
        f"Couldn't find {pd_predictions_path}. "
        "Run Notebook 04 (`04_transfer_gap_measure.ipynb`) to completion — "
        "Section 5 writes pd_predictions.parquet to this location."
    )

pd_predictions = pd.read_parquet(pd_predictions_path)

# Validate the columns this notebook actually depends on.
expected_columns = {"host", "true_category", "correct_at_category"}
missing = expected_columns - set(pd_predictions.columns)
if missing:
    raise KeyError(
        f"pd_predictions.parquet is missing these columns: {sorted(missing)}. "
        f"Re-run Notebook 04 to regenerate the file."
    )

print(f"Loaded: {pd_predictions_path}")
print(f"  Shape: {pd_predictions.shape[0]} rows × {pd_predictions.shape[1]} columns")
print(f"  Columns: {list(pd_predictions.columns)}")

In [ ]:
### 2.3) Cherry-pick fields for downstream sections

# Sections 3 and 4 (host cut, category cut) use these.
n_bootstrap = precommit["statistical_test"]["n_bootstrap"]
alpha = precommit["statistical_test"]["alpha"]

# Section 5 (figures and summary) uses this for category ordering.
categories_used = precommit["class_space_remapping"]["categories_used"]

# Apply the DEV_MODE bootstrap cap if it's set.
if BOOTSTRAP_CAP is not None:
    if n_bootstrap > BOOTSTRAP_CAP:
        print(f"DEV_MODE: capping n_bootstrap from {n_bootstrap} to {BOOTSTRAP_CAP}")
        n_bootstrap = BOOTSTRAP_CAP

print("Pre-commits loaded:")
print(f"  Bootstrap iterations:  {n_bootstrap}")
print(f"  Significance level:    α = {alpha}")
print(f"  Categories committed:  {len(categories_used)} ({categories_used})")

Three notes on what these values commit you to and what they don't.

**The predictions table is the load-bearing input.** Every cut in
Sections 3 and 4 starts from `pd_predictions`. The host cut groups
on `host`; the category cut groups on `true_category`; both compute
accuracy from `correct_at_category`. If N04 ran with a different
precommit than the one loaded here — say, you re-ran N02 between
runs and the category mapping shifted — the predictions table
reflects N04's run, and this notebook's results follow the
predictions, not the precommit. Section 5's summary JSON captures
both so any divergence is visible.

**`categories_used` is informational with one operational use.** This
notebook reads it so the per-category bar chart can order its bars
consistently across runs. It's also written into the summary JSON
as methodological provenance. The analysis itself doesn't depend on
it — if `categories_used` and the categories present in the
predictions table disagree, the chart shows what's in the data.

**Everything else in `precommit.json` is informational here.** The
class-space mappings were already applied upstream in N04, so this
notebook doesn't re-apply them. The aggregation level and statistical
test choice are recorded into the summary JSON for provenance, but
this notebook ships with a fixed approach (percentile bootstrap CIs
on per-group accuracy) and doesn't branch on those fields.